# EviChange: 4b experiments (cross-validation, Dynamic World evidence, other models)

One notebook, run in **parts**. Each part fits in one Colab session. Set `PART` in cell 0, then **Runtime → Run all**.
Keep the tab open (Colab Pro stops when the browser closes). Each part downloads `exp_<PART>.zip` at the end:
**send it to P2** before starting the next part.

| Part | What | Runs | Time (A100) | Units |
|---|---|---|---|---|
| **A** | Dynamic World as evidence, untrained: Qwen2.5-VL-7B and Qwen3-VL-8B on the 80 held-out boxes | 2 | ~0.5 h | ~6 |
| **B1** | Cross-validation, plain prompt: change weight 1 and 3, 3 folds of the 327 training boxes | 6 | ~4 h | ~50 |
| **B2** | Cross-validation, Dynamic World prompt: change weight 1 and 3, 3 folds | 6 | ~4 h | ~50 |
| **C** | The chosen setting on all 327 boxes, 3 seeds, each scored once on the held-out boxes | 3 | ~2.5 h | ~30 |
| **D** | The chosen setting on Qwen3-VL-8B and InternVL3-8B, scored on the held-out boxes | 2 | ~2 h | ~25 |
| **C2** | Split design check: untrained Qwen2.5-VL-7B with Dynamic World on the 327 training boxes, and Qwen3-VL-8B (chosen setting) in 3-fold cross-validation | 4 | ~1–1.5 h | ~15 |
| **E** | The chosen setting on Qwen2.5-VL-32B (4-bit, QLoRA), scored on the held-out boxes | 1 | ~3–4 h | ~45 |

**C2** (added 10 Nov) checks, on the training boxes only, the split design seen on the held-out boxes: main cover from a fine-tune, change from the untrained model with Dynamic World evidence. Run it **before E**.

Times are estimates; each run prints how long it took. **C, D and E wait for P2** to pick the setting from B1 + B2.

**How the setting is chosen (fixed before any result, `docs/analysis_plan.md`):** the setting with the highest mean
cross-validation balanced accuracy of the derived change; if two are within 0.02, the higher mean main-cover accuracy.
The held-out boxes are never used to choose.

**Files to upload** (cell 3), from `D:\EviChange\data\finetune\`: `chips.zip`, `cover_train.jsonl`,
`cover_holdout.jsonl`, `cover_dw_train.jsonl`, `cover_dw_holdout.jsonl`.

In [ ]:
PART = "C2"           # "A", "B1", "B2", "C", "D", "C2" or "E"

# Set by P2 after B1 + B2 (used by C, D and E):
CHOSEN = {"variant": "dw", "change_weight": 3}

## 1. Install, and check the GPU

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.57" peft accelerate pillow bitsandbytes

import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {mem:.0f} GB")
assert mem > 35, "Needs an A100 (40 GB or more)"

## 2. Get the code

In [ ]:
!rm -rf /content/EviChange && git clone -q --depth 1 https://github.com/nachikethpt/EviChange /content/EviChange
!grep -c "split_scores" /content/EviChange/scripts/finetune_train.py

## 3. Upload the data

Click **Choose files** and pick the **5 files** from `D:\EviChange\data\finetune\` (hold Ctrl):
`chips.zip`, `cover_train.jsonl`, `cover_holdout.jsonl`, `cover_dw_train.jsonl`, `cover_dw_holdout.jsonl`.

In [ ]:
import os, zipfile
from pathlib import Path
from google.colab import files

FT = Path("/content/ft"); FT.mkdir(exist_ok=True)
os.chdir(FT)
NEEDED = ["chips.zip", "cover_train.jsonl", "cover_holdout.jsonl", "cover_dw_train.jsonl", "cover_dw_holdout.jsonl"]
missing = [f for f in NEEDED if not (FT / f).exists()]
if missing:
    print("pick:", ", ".join(missing))
    files.upload()
for p in FT.glob("* (*)*"):                      # a re-upload is saved as "name (1).ext"
    p.replace(FT / (p.stem.split(" (")[0] + p.suffix))
missing = [f for f in NEEDED if not (FT / f).exists()]
assert not missing, f"still missing: {missing}"
with zipfile.ZipFile(FT / "chips.zip") as z:
    z.extractall(FT)
count = lambda f: sum(1 for _ in open(FT / f, encoding="utf-8"))
sizes = {f: count(f) for f in NEEDED[1:]}
print(sizes)
assert sizes == {"cover_train.jsonl": 327, "cover_holdout.jsonl": 80, "cover_dw_train.jsonl": 327, "cover_dw_holdout.jsonl": 80}, \
    "wrong file sizes: check that you picked cover_*.jsonl, not check_*.jsonl"

## 4. Run the part

Each run streams its log here and saves its score. If the session drops, start again with the same `PART`:
runs that already finished in this session are skipped, earlier sessions' runs are not (download each part's zip).

In [ ]:
import json, shutil, subprocess, sys, time
from pathlib import Path

FT = Path("/content/ft"); RES = Path(f"/content/exp_{PART}"); RES.mkdir(exist_ok=True)
SCRIPT = "/content/EviChange/scripts/finetune_train.py"
Q25, Q3, IVL, Q32 = "Qwen/Qwen2.5-VL-7B-Instruct", "Qwen/Qwen3-VL-8B-Instruct", "OpenGVLab/InternVL3-8B-hf", "Qwen/Qwen2.5-VL-32B-Instruct"

def run(tag, args):
    # One run: stream its log, keep the SCORE lines, save them to RES/scores.jsonl.
    if (RES / f"{tag}.done").exists():
        print(f"--- {tag}: already done in this session"); return
    print(f"=== {tag} ===", flush=True)
    t0 = time.time(); scores = []
    with open(RES / f"{tag}.log", "w") as log:
        proc = subprocess.Popen([sys.executable, SCRIPT] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, cwd="/content/EviChange")
        for line in proc.stdout:
            log.write(line)
            if "SCORE" in line:
                scores.append(json.loads(line.split("SCORE ", 1)[1]))
            if not line.startswith(("Loading", "Fetching", "Downloading")) and "it/s]" not in line:
                print(line, end="", flush=True)
        ok = proc.wait() == 0
    rec = {"tag": tag, "args": args, "ok": ok, "minutes": round((time.time() - t0) / 60, 1), "scores": scores}
    with open(RES / "scores.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")
    if ok:
        (RES / f"{tag}.done").touch()
    print(f"--- {tag}: {'ok' if ok else 'FAILED'} in {rec['minutes']} min", flush=True)

def train(tag, base, variant, weight, seed=17, folds=0, fold=0, four_bit=False, holdout=False):
    out = Path("/content/runs") / tag
    a = ["train", "--task", "cover", "--data", str(FT), "--out", str(out), "--base", base,
         "--change_weight", str(weight), "--seed", str(seed)]
    a += ["--variant", variant] if variant else []
    a += ["--folds", str(folds), "--fold", str(fold)] if folds else []
    a += ["--four_bit"] if four_bit else []
    a += ["--eval_holdout"] if holdout else []
    run(tag, a)
    for name in ("fold_score.json", "holdout_score.json", "train_config.json"):    # keep scores, not adapters
        src = out / "cover" / name
        if src.exists():
            shutil.copy(src, RES / f"{tag}__{name}")
    for p in (out / "cover").glob("predictions_*.jsonl"):
        shutil.copy(p, RES / f"{tag}__{p.name}")
    shutil.rmtree(out, ignore_errors=True)                                          # free the disk

if PART == "A":
    for base in (Q25, Q3):
        tag = base.split("/")[-1] + "_dw_zero_shot"
        run(tag, ["eval", "--task", "cover", "--data", str(FT), "--base-only", "--base", base, "--variant", "dw"])
        for p in FT.glob("predictions_cover_*_dw.jsonl"):
            shutil.copy(p, RES / p.name)
elif PART in ("B1", "B2"):
    variant = "" if PART == "B1" else "dw"
    for weight in (1, 3):
        for fold in range(3):
            train(f"cv_{variant or 'plain'}_w{weight}_fold{fold}", Q25, variant, weight, folds=3, fold=fold)
elif PART == "C":
    for seed in (17, 18, 19):
        train(f"final_{CHOSEN['variant'] or 'plain'}_w{CHOSEN['change_weight']}_seed{seed}", Q25,
              CHOSEN["variant"], CHOSEN["change_weight"], seed=seed, holdout=True)
elif PART == "D":
    for base in (Q3, IVL):
        train(f"{base.split('/')[-1]}_{CHOSEN['variant'] or 'plain'}_w{CHOSEN['change_weight']}", base,
              CHOSEN["variant"], CHOSEN["change_weight"], holdout=True)
elif PART == "C2":
    # (1) the untrained model with Dynamic World evidence answers the 327 training boxes (it was never trained on them)
    run("Qwen2.5-VL-7B-Instruct_dw_zero_shot_train",
        ["eval", "--task", "cover", "--data", str(FT), "--base-only", "--base", Q25, "--variant", "dw", "--split", "train"])
    for p in FT.glob("predictions_cover_*_dw_train.jsonl"):
        shutil.copy(p, RES / p.name)
    # (2) Qwen3-VL-8B with the chosen setting, 3-fold cross-validation (same folds as B1 / B2)
    for fold in range(3):
        train(f"cv_q3_{CHOSEN['variant'] or 'plain'}_w{CHOSEN['change_weight']}_fold{fold}", Q3,
              CHOSEN["variant"], CHOSEN["change_weight"], folds=3, fold=fold)
    # (3) the split design on the folds: Qwen3-VL-8B's covers, the untrained model's change verdicts
    folds = sorted(str(p) for p in RES.glob("cv_q3_*_fold*__predictions_fold.jsonl"))
    if len(folds) == 3:
        r = subprocess.run([sys.executable, "scripts/finetune_split_cv.py", "--data", str(FT), "--cover", *folds,
                        "--change", str(RES / "predictions_cover_base_dw_train.jsonl"),
                        "--out", str(RES / "split_cv_q3.json")], cwd="/content/EviChange",
                       capture_output=True, text=True)
        print(r.stdout, r.stderr)
elif PART == "E":
    train(f"qwen2_5-vl-32b_qlora_{CHOSEN['variant'] or 'plain'}_w{CHOSEN['change_weight']}", Q32,
          CHOSEN["variant"], CHOSEN["change_weight"], four_bit=True, holdout=True)
else:
    raise SystemExit(f"unknown PART {PART!r}")

print()
for line in open(RES / "scores.jsonl"):
    r = json.loads(line)
    s = (r["scores"] or [{}])[-1]
    print(f"{r['tag']}: {'ok' if r['ok'] else 'FAILED'}, {r['minutes']} min, main cover {s.get('main_cover_accuracy')}, "
          f"change balanced {(s.get('derived_change') or {}).get('balanced')}")

## 5. Download the results

This downloads `exp_<PART>.zip` (scores, predictions and logs; no adapters). **Send it to P2.**

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"/content/exp_{PART}", "zip", f"/content/exp_{PART}")
files.download(f"/content/exp_{PART}.zip")